# Proyecto Iris — Programación Orientada a Objetos

**Asignatura:** Programación II  
**Unidad 1:** Programación Orientada a Objetos en Python

Este notebook construye, paso a paso, un sistema en Python que representa las muestras del archivo `Iris.csv` usando **Programación Orientada a Objetos (POO)**.

## ¿Qué vamos a hacer?

Vamos a representar **flores Iris** como objetos. Cada flor tiene:
- Un **Id** (número único)
- 4 medidas: largo y ancho del **sépalo**, largo y ancho del **pétalo**
- Una **especie**: `setosa`, `versicolor` o `virginica`

## ¿Cómo está organizado este notebook?

Sigue el orden del sílabo:

**Tema 1 — Modelado base:**
1. Clase `MuestraIris` (representa una flor)
2. Encapsulamiento (proteger los datos)
3. Clase `ColeccionIris` (manejar muchas flores)

**Tema 2 — Ampliación:**
4. Subclases por especie (herencia)
5. Polimorfismo y `super()`
6. Métodos mágicos y sobrecarga de operadores
7. Patrón Singleton
8. Demostración final

**Importante:** ejecuta las celdas **en orden**, porque cada una depende de las anteriores.

---
## Paso 0: Verificar el archivo Iris.csv

Antes de programar, verificamos que el archivo `Iris.csv` esté disponible. Asegúrate de que esté en la **misma carpeta** que este notebook, o ajusta la ruta si lo tienes en otro lado.

In [ ]:
# Importamos csv (módulo estándar de Python para leer archivos CSV)
import csv

# Ruta del archivo. Ajústala si lo tienes en otra carpeta.
RUTA_CSV = "Iris.csv"

# Vamos a abrir el archivo y mostrar las primeras 3 filas para verificar
with open(RUTA_CSV, encoding="utf-8") as archivo:
    reader = csv.DictReader(archivo)  # DictReader lee cada fila como un diccionario
    for i, fila in enumerate(reader):
        if i >= 3:  # solo las primeras 3
            break
        print(fila)

print("\n✅ El archivo se lee correctamente.")

### ¿Qué viste arriba?

Cada fila se imprime como un diccionario con estas claves:
- `'Id'`: identificador único (1, 2, 3...)
- `'SepalLengthCm'`: largo del sépalo en cm
- `'SepalWidthCm'`: ancho del sépalo en cm
- `'PetalLengthCm'`: largo del pétalo en cm
- `'PetalWidthCm'`: ancho del pétalo en cm
- `'Species'`: especie de la flor (`Iris-setosa`, `Iris-versicolor` o `Iris-virginica`)

**Observación importante:** todos los valores vienen como **texto** (strings). Cuando los usemos, tendremos que convertir los números a `int` o `float`.

---
# 🎯 TEMA 1: Modelado base

## Paso 1: Crear la clase `MuestraIris`

Una **clase** es un molde para crear objetos. Vamos a crear la clase `MuestraIris` que será el molde para todas las flores del dataset.

### Conceptos que vamos a usar:

- **`class`**: palabra clave para definir una clase.
- **`__init__`**: método especial que se ejecuta cuando creas un objeto nuevo. Sirve para inicializar sus datos.
- **`self`**: representa al objeto actual. Es como decir "yo mismo". Cada objeto tiene su propio `self`.
- **Atributos**: los datos que guarda el objeto (ej: `self._id`, `self._sepal_length`).
- **Atributos privados**: cuando ponemos un guion bajo (`_`) al inicio del nombre, indicamos que ese atributo **no debe modificarse desde fuera**. Es una convención para protegerlo.
- **Métodos**: las funciones que el objeto sabe hacer (ej: `describir()`, `area_sepalo_aprox()`).

Empecemos con una versión **muy simple**, sin validaciones todavía. En el próximo paso le agregamos protección.

In [ ]:
# Versión simple de MuestraIris (sin validaciones todavía)

class MuestraIrisSimple:
    """Versión inicial: solo guarda los datos, sin protegerlos."""
    
    # __init__ se ejecuta automáticamente cuando creas un objeto nuevo.
    # Recibe todos los datos que necesita la flor.
    def __init__(self, id_muestra, sepal_length, sepal_width,
                 petal_length, petal_width, especie):
        # 'self' es el objeto que se está creando.
        # Aquí guardamos cada dato como un atributo del objeto.
        self.id = id_muestra
        self.sepal_length = sepal_length
        self.sepal_width = sepal_width
        self.petal_length = petal_length
        self.petal_width = petal_width
        self.especie = especie
    
    # Un método: una función que pertenece a la clase.
    # Siempre recibe 'self' como primer parámetro.
    def describir(self):
        """Devuelve una descripción legible de la flor."""
        return f"Flor #{self.id}: especie={self.especie}, sépalo={self.sepal_length}x{self.sepal_width}"


# Probemos la clase: creamos una flor
flor1 = MuestraIrisSimple(1, 5.1, 3.5, 1.4, 0.2, "Iris-setosa")

# Accedemos a sus atributos
print("Id:", flor1.id)
print("Largo sépalo:", flor1.sepal_length)
print("Especie:", flor1.especie)

# Llamamos a su método
print("Descripción:", flor1.describir())

### ¡Felicidades! Acabas de crear tu primer objeto.

Ahora `flor1` es un objeto con sus propios datos y métodos. Si crearas otro objeto:

```python
flor2 = MuestraIrisSimple(2, 4.9, 3.0, 1.4, 0.2, "Iris-setosa")
```

...sería un objeto **independiente**, con sus propios datos.

### Pero hay un problema...

La versión actual no protege los datos. Mira lo que podría pasar:

In [ ]:
# Esto NO debería ser posible: una medida negativa no tiene sentido
flor1.sepal_length = -50
print("Largo sépalo:", flor1.sepal_length)  # ¡Aceptó un valor absurdo!

# Esto tampoco: una especie inventada
flor1.especie = "tulipan"
print("Especie:", flor1.especie)  # ¡También lo aceptó!

# Necesitamos PROTEGER los datos. Eso es ENCAPSULAMIENTO.

---
## Paso 2: Encapsulamiento — Proteger los datos

**Encapsulamiento** significa: ocultar los datos internos y obligar a que se accedan a través de "puertas" controladas (getters y setters).

### ¿Cómo se hace en Python?

1. **Atributos privados:** se nombran con guion bajo, como `self._sepal_length`. Por convención, *no se tocan desde fuera*.
2. **`@property`** (getter): permite **leer** el atributo como si fuera público.
3. **`@nombre.setter`** (setter): permite **modificar** el atributo, pero pasando primero por una validación.

Es como un guardia en la puerta: solo deja pasar valores correctos.

Esta vez creamos la versión **definitiva** de `MuestraIris`, con todas las protecciones.

In [ ]:
class MuestraIris:
    """Representa una muestra del conjunto de datos Iris (versión con encapsulamiento)."""
    
    # ---------------------------------------------------------------- #
    # Atributo de CLASE (no de instancia): es compartido por todas las 
    # muestras. Se usa para validar que la especie esté en este conjunto.
    # ---------------------------------------------------------------- #
    ESPECIES_VALIDAS = {"setosa", "versicolor", "virginica"}
    
    # ---------------------------------------------------------------- #
    # Constructor
    # ---------------------------------------------------------------- #
    def __init__(self, id_muestra, sepal_length, sepal_width,
                 petal_length, petal_width, especie):
        """
        Inicializa una muestra. Cada parámetro pasa por validación
        antes de guardarse.
        """
        # El Id es ESPECIAL: se asigna directamente al atributo privado
        # porque NO queremos que se pueda modificar después de la creación.
        # (No le pondremos setter, así que será de "solo lectura".)
        self._validar_id(id_muestra)
        self._id = int(id_muestra)
        
        # Las medidas y la especie se asignan usando los SETTERS,
        # que validan automáticamente. Por eso usamos 'self.sepal_length'
        # (sin guion bajo) en vez de 'self._sepal_length'.
        self.sepal_length = sepal_length
        self.sepal_width = sepal_width
        self.petal_length = petal_length
        self.petal_width = petal_width
        self.especie = especie
    
    # ---------------------------------------------------------------- #
    # Propiedad de SOLO LECTURA: el Id no se puede cambiar
    # ---------------------------------------------------------------- #
    @property
    def id(self):
        """Identificador único. No tiene setter: es inmutable."""
        return self._id
    
    # ---------------------------------------------------------------- #
    # Propiedades con validación: largo del sépalo
    # ---------------------------------------------------------------- #
    
    # @property convierte este método en un "getter":
    # cuando alguien escribe 'flor.sepal_length', en realidad se ejecuta este código.
    @property
    def sepal_length(self):
        return self._sepal_length
    
    # @sepal_length.setter convierte este método en un "setter":
    # cuando alguien escribe 'flor.sepal_length = 5.1', se ejecuta este código.
    @sepal_length.setter
    def sepal_length(self, valor):
        # Primero validamos
        self._validar_medida(valor, "sepal_length")
        # Si pasó la validación, guardamos
        self._sepal_length = float(valor)
    
    # ---------------------------------------------------------------- #
    # Las otras tres medidas siguen el mismo patrón
    # ---------------------------------------------------------------- #
    
    @property
    def sepal_width(self):
        return self._sepal_width
    
    @sepal_width.setter
    def sepal_width(self, valor):
        self._validar_medida(valor, "sepal_width")
        self._sepal_width = float(valor)
    
    @property
    def petal_length(self):
        return self._petal_length
    
    @petal_length.setter
    def petal_length(self, valor):
        self._validar_medida(valor, "petal_length")
        self._petal_length = float(valor)
    
    @property
    def petal_width(self):
        return self._petal_width
    
    @petal_width.setter
    def petal_width(self, valor):
        self._validar_medida(valor, "petal_width")
        self._petal_width = float(valor)
    
    # ---------------------------------------------------------------- #
    # Propiedad ESPECIE: caso especial porque acepta dos formatos
    # ---------------------------------------------------------------- #
    @property
    def especie(self):
        return self._especie
    
    @especie.setter
    def especie(self, valor):
        # Verifica que sea texto
        if not isinstance(valor, str):
            raise ValueError("La especie debe ser texto.")
        
        # Normalizamos: minúsculas, sin espacios
        valor_norm = valor.strip().lower()
        
        # Si viene como 'iris-setosa', le quitamos el prefijo 'iris-'
        # (porque internamente queremos guardar solo 'setosa')
        if valor_norm.startswith("iris-"):
            valor_norm = valor_norm[5:]  # [5:] significa: desde el carácter 5 en adelante
        
        # Verifica que sea una especie válida
        if valor_norm not in self.ESPECIES_VALIDAS:
            raise ValueError(
                f"Especie '{valor}' no válida. "
                f"Debe ser: {sorted(self.ESPECIES_VALIDAS)}"
            )
        
        # Si pasó todo, guardamos
        self._especie = valor_norm
    
    # ---------------------------------------------------------------- #
    # Métodos privados de validación (uso interno)
    # ---------------------------------------------------------------- #
    
    # @staticmethod indica que este método NO usa 'self':
    # es como una función normal pero pertenece a la clase.
    @staticmethod
    def _validar_id(valor):
        """Valida que el Id sea un entero positivo."""
        # bool es subclase de int en Python; lo descartamos explícitamente
        if isinstance(valor, bool) or not isinstance(valor, int):
            raise ValueError(f"El Id debe ser entero, se recibió {type(valor).__name__}.")
        if valor <= 0:
            raise ValueError(f"El Id debe ser positivo, se recibió {valor}.")
    
    @staticmethod
    def _validar_medida(valor, nombre):
        """Valida que una medida sea numérica y positiva."""
        if isinstance(valor, bool) or not isinstance(valor, (int, float)):
            raise ValueError(f"'{nombre}' debe ser numérico.")
        if valor <= 0:
            raise ValueError(f"'{nombre}' debe ser mayor que 0, se recibió {valor}.")
    
    # ---------------------------------------------------------------- #
    # Métodos públicos (lo que la flor sabe hacer)
    # ---------------------------------------------------------------- #
    
    def describir(self):
        """Devuelve una descripción legible."""
        return (f"[#{self._id}] Iris-{self._especie} | "
                f"sépalo: {self._sepal_length} x {self._sepal_width} cm | "
                f"pétalo: {self._petal_length} x {self._petal_width} cm")
    
    def area_sepalo_aprox(self):
        """Área aproximada del sépalo (largo x ancho)."""
        return self._sepal_length * self._sepal_width
    
    def area_petalo_aprox(self):
        """Área aproximada del pétalo (largo x ancho)."""
        return self._petal_length * self._petal_width
    
    def resumen(self):
        """Devuelve un diccionario con todos los atributos."""
        return {
            "id": self._id,
            "sepal_length": self._sepal_length,
            "sepal_width": self._sepal_width,
            "petal_length": self._petal_length,
            "petal_width": self._petal_width,
            "especie": self._especie,
        }


print("✅ Clase MuestraIris definida correctamente.")

### Probemos la nueva clase con encapsulamiento

In [ ]:
# Caso normal: creamos una flor con datos válidos
flor = MuestraIris(1, 5.1, 3.5, 1.4, 0.2, "Iris-setosa")
print(flor.describir())
print("Especie guardada:", flor.especie)  # se guardó como 'setosa' (sin el prefijo)
print("Área del sépalo:", flor.area_sepalo_aprox(), "cm²")
print("Resumen:", flor.resumen())

In [ ]:
# Ahora probamos las validaciones: deben RECHAZAR datos malos

print("Intento 1: poner sepal_length = -3 (negativo)")
try:
    flor.sepal_length = -3
except ValueError as error:
    print(f"  ❌ Rechazado: {error}")

print("\nIntento 2: especie inválida 'tulipan'")
try:
    flor.especie = "tulipan"
except ValueError as error:
    print(f"  ❌ Rechazado: {error}")

print("\nIntento 3: modificar el Id (es de solo lectura)")
try:
    flor.id = 999
except AttributeError as error:
    print(f"  ❌ Rechazado: el Id es inmutable.")

print("\nIntento 4: crear flor con texto en una medida")
try:
    MuestraIris(99, "hola", 3.0, 1.5, 0.3, "setosa")
except ValueError as error:
    print(f"  ❌ Rechazado: {error}")

### ¡Excelente! El encapsulamiento funciona

Los datos están protegidos:
- No se aceptan medidas negativas o no numéricas.
- No se aceptan especies fuera de la lista válida.
- El Id no se puede modificar después de crear la flor.

Esto es **encapsulamiento bien hecho**: el objeto controla su propio estado y nadie puede dejarlo en un estado inválido.

---
## Paso 3: Clase `ColeccionIris` — Manejar muchas flores

Hasta ahora podemos crear flores una por una. Pero el dataset tiene **150 flores**. Necesitamos una forma de manejarlas en grupo.

Por eso creamos `ColeccionIris`: una clase que guarda una lista de muestras y permite:
- Agregar una muestra
- Cargar todas desde el archivo CSV
- Filtrar por especie
- Buscar por Id
- Calcular promedios

In [ ]:
class ColeccionIris:
    """Administra una colección de muestras del dataset Iris."""
    
    def __init__(self):
        # La colección empieza vacía: una lista interna donde guardaremos las flores
        self._muestras = []
    
    def agregar(self, muestra):
        """Agrega una muestra a la colección."""
        # isinstance verifica que el objeto sea una MuestraIris.
        # Así evitamos que alguien meta cualquier cosa en la colección.
        if not isinstance(muestra, MuestraIris):
            raise TypeError("Solo se pueden agregar objetos MuestraIris.")
        self._muestras.append(muestra)
    
    def cargar_desde_csv(self, ruta):
        """Lee Iris.csv y crea una muestra por cada fila."""
        with open(ruta, encoding="utf-8") as archivo:
            reader = csv.DictReader(archivo)
            for fila in reader:
                # Cada fila es un diccionario con los datos del CSV
                muestra = MuestraIris(
                    id_muestra=int(fila["Id"]),
                    sepal_length=float(fila["SepalLengthCm"]),
                    sepal_width=float(fila["SepalWidthCm"]),
                    petal_length=float(fila["PetalLengthCm"]),
                    petal_width=float(fila["PetalWidthCm"]),
                    especie=fila["Species"],
                )
                self.agregar(muestra)
    
    def total(self):
        """Cantidad de muestras en la colección."""
        return len(self._muestras)
    
    def filtrar_por_especie(self, especie):
        """Devuelve una lista con las muestras de la especie indicada."""
        # Normalizamos el nombre igual que en el setter de la clase
        nombre = especie.strip().lower().replace("iris-", "")
        # Esta es una "list comprehension": forma compacta de filtrar listas
        return [m for m in self._muestras if m.especie == nombre]
    
    def obtener_por_id(self, id_buscado):
        """Busca una muestra por su Id. Devuelve None si no existe."""
        for m in self._muestras:
            if m.id == id_buscado:
                return m
        return None
    
    def promedio_petalo(self):
        """Promedio del largo del pétalo en toda la colección."""
        if not self._muestras:  # si la lista está vacía
            return 0
        total = sum(m.petal_length for m in self._muestras)
        return total / len(self._muestras)


print("✅ Clase ColeccionIris definida correctamente.")

### Probemos la colección cargando el CSV completo

In [ ]:
# Creamos la colección y la cargamos desde el archivo
coleccion = ColeccionIris()
coleccion.cargar_desde_csv(RUTA_CSV)

print(f"Total de muestras cargadas: {coleccion.total()}")
print(f"Setosas:    {len(coleccion.filtrar_por_especie('setosa'))}")
print(f"Versicolor: {len(coleccion.filtrar_por_especie('versicolor'))}")
print(f"Virginica:  {len(coleccion.filtrar_por_especie('virginica'))}")
print(f"Promedio largo de pétalo: {coleccion.promedio_petalo():.2f} cm")

# Buscamos la muestra con Id=50
m50 = coleccion.obtener_por_id(50)
print(f"\nMuestra con Id=50:")
print(" ", m50.describir())

### ¡Tema 1 completado!

Hasta aquí cubrimos los contenidos del **Tema 1** del sílabo:
- ✅ Clases y objetos (`MuestraIris`, `ColeccionIris`)
- ✅ Atributos y métodos
- ✅ Encapsulamiento (atributos privados con `_`)
- ✅ Métodos de acceso (`@property` y setters con validación)

Ahora vamos al **Tema 2**, donde extendemos el sistema.

---
# 🚀 TEMA 2: Ampliación del sistema

## Paso 4: Herencia — Subclases por especie

Como hay **tres especies** distintas (setosa, versicolor, virginica), tiene sentido crear una **subclase para cada una**. Cada subclase **hereda** todo lo de `MuestraIris` y puede agregar comportamiento propio.

### ¿Qué es la herencia?

Es un mecanismo donde una clase "hija" (subclase) toma todo lo de una clase "padre" (superclase) y puede:
- **Reutilizar** sus métodos y atributos.
- **Sobrescribir** algunos para que se comporten distinto.
- **Agregar** nuevos.

**Sintaxis:** `class Hija(Padre):` significa "Hija hereda de Padre".

### ¿Qué es `super()`?

Cuando una subclase quiere llamar a un método del padre (especialmente `__init__`), usa `super()`. Esto evita repetir código.

### ¿Qué es polimorfismo?

Es cuando varias clases tienen un método con el **mismo nombre** pero **comportamiento distinto**. Por ejemplo, las tres especies tendrán un método `clasificar_tamano()`, pero cada una usará umbrales propios.

In [ ]:
# La sintaxis 'class IrisSetosa(MuestraIris)' indica que IrisSetosa
# HEREDA de MuestraIris: tiene todos sus atributos y métodos automáticamente.

class IrisSetosa(MuestraIris):
    """Iris-setosa: especie pequeña, pétalos cortos."""
    
    def __init__(self, id_muestra, sepal_length, sepal_width, petal_length, petal_width):
        # super() llama al __init__ de la clase padre (MuestraIris).
        # Le pasamos todos los datos, FIJANDO la especie como 'setosa'.
        # Así, al crear un IrisSetosa, no hace falta indicar la especie.
        super().__init__(id_muestra, sepal_length, sepal_width,
                         petal_length, petal_width, especie="setosa")
    
    # Método propio de la subclase (no existe en el padre)
    def caracteristicas_especie(self):
        return "Iris-setosa: especie pequeña, fácil de distinguir, pétalos cortos."
    
    # Este método tendrá la misma firma en las tres subclases,
    # pero con comportamiento distinto. Eso es POLIMORFISMO.
    def clasificar_tamano(self):
        if self.petal_length < 1.5:
            return "pequeña"
        return "grande para setosa"


class IrisVersicolor(MuestraIris):
    """Iris-versicolor: especie de tamaño intermedio."""
    
    def __init__(self, id_muestra, sepal_length, sepal_width, petal_length, petal_width):
        super().__init__(id_muestra, sepal_length, sepal_width,
                         petal_length, petal_width, especie="versicolor")
    
    def caracteristicas_especie(self):
        return "Iris-versicolor: especie intermedia entre setosa y virginica."
    
    def clasificar_tamano(self):
        # Umbrales DISTINTOS a los de setosa: eso es polimorfismo
        if self.petal_length < 4.0:
            return "pequeña para versicolor"
        if self.petal_length < 4.8:
            return "mediana"
        return "grande para versicolor"


class IrisVirginica(MuestraIris):
    """Iris-virginica: especie grande, pétalos largos."""
    
    def __init__(self, id_muestra, sepal_length, sepal_width, petal_length, petal_width):
        super().__init__(id_muestra, sepal_length, sepal_width,
                         petal_length, petal_width, especie="virginica")
    
    def caracteristicas_especie(self):
        return "Iris-virginica: especie grande, pétalos largos y anchos."
    
    def clasificar_tamano(self):
        if self.petal_length < 5.0:
            return "pequeña para virginica"
        if self.petal_length < 5.7:
            return "mediana"
        return "grande"


print("✅ Tres subclases definidas: IrisSetosa, IrisVersicolor, IrisVirginica.")

### Probemos la herencia y el polimorfismo

In [ ]:
# Creamos una flor de cada especie
setosa = IrisSetosa(1, 5.1, 3.5, 1.4, 0.2)
versicolor = IrisVersicolor(51, 7.0, 3.2, 4.7, 1.4)
virginica = IrisVirginica(101, 6.3, 3.3, 6.0, 2.5)

# Cada una HEREDÓ los métodos del padre, como describir()
for flor in [setosa, versicolor, virginica]:
    print(flor.describir())

print()

# Y cada una tiene comportamiento PROPIO en clasificar_tamano() (polimorfismo)
for flor in [setosa, versicolor, virginica]:
    print(f"{type(flor).__name__:15} -> {flor.clasificar_tamano()}")

### ¿Notaste lo importante?

El bucle `for` llama a `clasificar_tamano()` sin saber qué tipo de flor es. **Cada flor responde a su manera**. Eso es polimorfismo: una misma operación, distintos comportamientos.

Además, ninguna de las subclases tuvo que reescribir `describir()`, `area_sepalo_aprox()` ni las validaciones. Todo eso lo **heredaron** de `MuestraIris`.

---
## Paso 4.1: Función "fábrica" `crear_muestra`

Ahora que tenemos subclases, queremos que al cargar el CSV se cree la subclase correcta según la especie. Para eso creamos una función que decide qué subclase instanciar.

In [ ]:
def crear_muestra(id_muestra, sepal_length, sepal_width,
                  petal_length, petal_width, especie):
    """
    Recibe los datos de una fila y devuelve la subclase correcta.
    
    Esta función se llama 'factory' (fábrica): decide qué tipo de objeto
    crear según un parámetro.
    """
    # Normalizamos el nombre (quitamos prefijo Iris- si lo tiene)
    nombre = especie.strip().lower().replace("iris-", "")
    
    if nombre == "setosa":
        return IrisSetosa(id_muestra, sepal_length, sepal_width,
                          petal_length, petal_width)
    elif nombre == "versicolor":
        return IrisVersicolor(id_muestra, sepal_length, sepal_width,
                              petal_length, petal_width)
    elif nombre == "virginica":
        return IrisVirginica(id_muestra, sepal_length, sepal_width,
                             petal_length, petal_width)
    else:
        raise ValueError(f"Especie desconocida: {especie}")


# Probemos la fábrica
ejemplo1 = crear_muestra(1, 5.1, 3.5, 1.4, 0.2, "Iris-setosa")
ejemplo2 = crear_muestra(51, 7.0, 3.2, 4.7, 1.4, "Iris-versicolor")

print(f"ejemplo1 es de tipo: {type(ejemplo1).__name__}")
print(f"ejemplo2 es de tipo: {type(ejemplo2).__name__}")

# isinstance verifica si un objeto es de cierta clase O DE ALGUNA DE SUS HIJAS
print(f"\n¿ejemplo1 es MuestraIris? {isinstance(ejemplo1, MuestraIris)}")
print(f"¿ejemplo1 es IrisSetosa? {isinstance(ejemplo1, IrisSetosa)}")
print(f"¿ejemplo1 es IrisVirginica? {isinstance(ejemplo1, IrisVirginica)}")

Fíjate: `ejemplo1` es a la vez `IrisSetosa` **y** `MuestraIris`. Eso es porque `IrisSetosa` hereda de `MuestraIris`. Una flor setosa **es también** una flor Iris.

---
## Paso 5: Métodos mágicos y sobrecarga de operadores

Los **métodos mágicos** (también llamados *dunder methods* por los doble guion bajo) son métodos especiales que Python llama **automáticamente** en ciertas situaciones.

### Los más comunes:

| Método | Cuándo se ejecuta | Ejemplo |
|---|---|---|
| `__str__` | `print(obj)` o `str(obj)` | texto legible |
| `__repr__` | inspección/debugging | texto técnico |
| `__eq__` | `a == b` | igualdad |
| `__lt__` | `a < b` | menor que |
| `__add__` | `a + b` | suma |
| `__hash__` | `hash(obj)` | identificación única |
| `__len__` | `len(obj)` | longitud |
| `__iter__` | `for x in obj` | iteración |
| `__getitem__` | `obj[i]` | acceso con corchetes |
| `__contains__` | `x in obj` | pertenencia |

Cuando defines estos métodos en tu clase, le das un significado a operadores como `+`, `==`, `<`. Esto es **sobrecarga de operadores**.

Vamos a crear una **versión mejorada de `MuestraIris`** que incluya estos métodos. La llamamos `MuestraIrisV2` para no perder la versión anterior, pero al final de este paso reemplazamos la original.

In [ ]:
# Vamos a AGREGAR métodos mágicos a la clase MuestraIris original
# usando una técnica: definir métodos sueltos y "pegarlos" a la clase.
# Es equivalente a haberlos puesto dentro de la clase desde el inicio.

def __str__(self):
    """Se ejecuta con print(flor). Devuelve texto legible."""
    return self.describir()

def __repr__(self):
    """Se ejecuta al inspeccionar el objeto. Devuelve texto técnico."""
    return (f"MuestraIris(id={self._id}, especie='{self._especie}', "
            f"sepal=({self._sepal_length}, {self._sepal_width}), "
            f"petal=({self._petal_length}, {self._petal_width}))")

def __eq__(self, otra):
    """Se ejecuta con flor1 == flor2."""
    # Si 'otra' no es una MuestraIris, devolvemos NotImplemented:
    # le decimos a Python "no sé comparar con esto".
    if not isinstance(otra, MuestraIris):
        return NotImplemented
    # Dos flores son iguales si todos sus datos coinciden
    return (self._sepal_length == otra._sepal_length and
            self._sepal_width == otra._sepal_width and
            self._petal_length == otra._petal_length and
            self._petal_width == otra._petal_width and
            self._especie == otra._especie)

def __lt__(self, otra):
    """Se ejecuta con flor1 < flor2. Permite ordenar."""
    if not isinstance(otra, MuestraIris):
        return NotImplemented
    # Ordenamos por largo del pétalo (criterio arbitrario)
    return self._petal_length < otra._petal_length

def __add__(self, otra):
    """Se ejecuta con flor1 + flor2. Devuelve una flor con los promedios."""
    if not isinstance(otra, MuestraIris):
        return NotImplemented
    # Solo permitimos sumar flores de la misma especie
    if self._especie != otra._especie:
        raise ValueError("Solo se pueden sumar muestras de la misma especie.")
    return MuestraIris(
        id_muestra=999,  # Id genérico para la flor promedio
        sepal_length=(self._sepal_length + otra._sepal_length) / 2,
        sepal_width=(self._sepal_width + otra._sepal_width) / 2,
        petal_length=(self._petal_length + otra._petal_length) / 2,
        petal_width=(self._petal_width + otra._petal_width) / 2,
        especie=self._especie,
    )

def __hash__(self):
    """Permite usar la flor como elemento de un set o clave de un dict."""
    return hash((self._id, self._especie))

# Ahora pegamos esos métodos a la clase MuestraIris
MuestraIris.__str__ = __str__
MuestraIris.__repr__ = __repr__
MuestraIris.__eq__ = __eq__
MuestraIris.__lt__ = __lt__
MuestraIris.__add__ = __add__
MuestraIris.__hash__ = __hash__

print("✅ Métodos mágicos agregados a MuestraIris.")

**Nota:** en un proyecto real, estos métodos irían **dentro** de la clase desde el inicio. Aquí los agregamos por separado solo para que veas claramente cuáles son los métodos mágicos y qué hace cada uno.

Ahora probemos que funcionan:

In [ ]:
# Creamos dos flores idénticas y una distinta
a = MuestraIris(1, 5.1, 3.5, 1.4, 0.2, "setosa")
b = MuestraIris(2, 5.1, 3.5, 1.4, 0.2, "setosa")  # mismos datos que 'a'
c = MuestraIris(3, 4.9, 3.0, 5.0, 1.5, "setosa")  # diferente

# __str__: print() llama a __str__ automáticamente
print("print(a):", a)

# __repr__: en una celda solo, el último valor se muestra con repr()
print("repr(a):", repr(a))

# __eq__: el operador == llama a __eq__
print("\n¿a == b? (mismos datos)", a == b)  # True
print("¿a == c? (diferentes)   ", a == c)  # False

# __lt__: el operador < llama a __lt__
print("\n¿a < c? (por largo de pétalo)", a < c)  # True (1.4 < 5.0)

# __add__: el operador + llama a __add__
promedio = a + c
print("\nFlor promedio (a + c):")
print(" ", promedio)

In [ ]:
# Como definimos __lt__, podemos ORDENAR una lista de flores con sorted()
flores = [
    MuestraIris(1, 5.1, 3.5, 5.0, 0.2, "setosa"),
    MuestraIris(2, 4.9, 3.0, 1.4, 0.2, "setosa"),
    MuestraIris(3, 5.0, 3.6, 3.0, 0.2, "setosa"),
]

print("Orden original (por Id):")
for f in flores:
    print(" ", f)

print("\nOrden ascendente por largo de pétalo (usando sorted):")
for f in sorted(flores):
    print(" ", f)

### Métodos mágicos en `ColeccionIris`

También podemos agregar métodos mágicos a la colección, para que se comporte **como una lista de Python**:
- `len(coleccion)` → cuenta cuántas flores tiene.
- `for flor in coleccion:` → recorre todas las flores.
- `coleccion[0]` → accede a la primera.
- `flor in coleccion` → verifica si está dentro.

In [ ]:
# Agregamos métodos mágicos a ColeccionIris (misma técnica)

def __len__(self):
    """Permite hacer len(coleccion)."""
    return len(self._muestras)

def __iter__(self):
    """Permite hacer 'for muestra in coleccion'."""
    return iter(self._muestras)

def __getitem__(self, indice):
    """Permite hacer 'coleccion[i]'."""
    return self._muestras[indice]

def __contains__(self, muestra):
    """Permite hacer 'muestra in coleccion'."""
    return muestra in self._muestras

def __str_col__(self):
    return f"ColeccionIris con {len(self._muestras)} muestras."

ColeccionIris.__len__ = __len__
ColeccionIris.__iter__ = __iter__
ColeccionIris.__getitem__ = __getitem__
ColeccionIris.__contains__ = __contains__
ColeccionIris.__str__ = __str_col__

print("✅ Métodos mágicos agregados a ColeccionIris.")

Ahora también modificaremos `cargar_desde_csv` para que use la **fábrica** y cree subclases en vez de `MuestraIris` plana:

In [ ]:
def cargar_desde_csv_v2(self, ruta):
    """Versión mejorada: usa la fábrica para crear subclases."""
    with open(ruta, encoding="utf-8") as archivo:
        reader = csv.DictReader(archivo)
        for fila in reader:
            muestra = crear_muestra(
                id_muestra=int(fila["Id"]),
                sepal_length=float(fila["SepalLengthCm"]),
                sepal_width=float(fila["SepalWidthCm"]),
                petal_length=float(fila["PetalLengthCm"]),
                petal_width=float(fila["PetalWidthCm"]),
                especie=fila["Species"],
            )
            self.agregar(muestra)

# Sobrescribimos el método anterior
ColeccionIris.cargar_desde_csv = cargar_desde_csv_v2
print("✅ ColeccionIris ahora carga subclases mediante la fábrica.")

In [ ]:
# Recargamos la colección para que use la nueva versión
coleccion = ColeccionIris()
coleccion.cargar_desde_csv(RUTA_CSV)

# Probemos los métodos mágicos:
print(f"len(coleccion): {len(coleccion)}")           # __len__
print(f"coleccion[0]:   {coleccion[0]}")             # __getitem__
print(f"coleccion[-1]:  {coleccion[-1]}")            # último elemento

# Iteración con for (usa __iter__)
print("\nPrimeras 3 muestras (con bucle for):")
for i, muestra in enumerate(coleccion):
    if i >= 3:
        break
    print(" ", muestra)

# Verificación con 'in' (usa __contains__)
primera = coleccion[0]
print(f"\n¿La primera muestra está en la colección? {primera in coleccion}")

# Verificación de tipos: ahora cada muestra es de su subclase
print(f"\nTipo de coleccion[0]:   {type(coleccion[0]).__name__}")
print(f"Tipo de coleccion[60]:  {type(coleccion[60]).__name__}")
print(f"Tipo de coleccion[120]: {type(coleccion[120]).__name__}")

¡Mira eso! La colección ahora se comporta como una lista (`len`, indexación, `for`, `in`) y cada muestra es de la subclase correcta según su especie.

---
## Paso 6: Patrón Singleton

El **Singleton** es un patrón de diseño donde una clase **solo puede tener una instancia** en todo el programa.

### ¿Por qué usarlo aquí?

Queremos un único "gestor" del dataset: que cargue el CSV una sola vez y que todas las partes del programa accedan a la misma colección. Si alguien crea otro `GestorDataset`, debe recibir la **misma instancia** que ya existe.

### ¿Cómo se implementa?

Sobrescribiendo `__new__`, que es el método que **crea el objeto** (se ejecuta antes de `__init__`).

1. Guardamos la única instancia en una variable de clase.
2. Cuando alguien hace `GestorDataset()`, primero verificamos si ya hay una.
3. Si ya hay → devolvemos esa.
4. Si no hay → creamos una nueva y la guardamos.

In [ ]:
class GestorDataset:
    """
    Gestor único del dataset Iris (patrón Singleton).
    
    Solo puede existir UNA instancia de esta clase. Si alguien intenta
    crear otra, recibe la misma instancia que ya existe.
    """
    
    # Variable de CLASE (compartida por todas las instancias).
    # Aquí guardamos la única instancia que existirá.
    _instancia = None
    
    def __new__(cls):
        """
        __new__ es el método que CREA el objeto. Se ejecuta antes de __init__.
        Aquí controlamos para que solo se cree una vez.
        
        - cls representa la clase (similar a self pero a nivel de clase)
        """
        # Si NO existe instancia previa, la creamos
        if cls._instancia is None:
            # super().__new__(cls) crea un objeto vacío de esta clase
            cls._instancia = super().__new__(cls)
            # Marca para saber que aún no se ha inicializado
            cls._instancia._inicializado = False
        # Devolvemos la instancia (nueva o la que ya existía)
        return cls._instancia
    
    def __init__(self):
        # Como __init__ se ejecuta CADA VEZ que alguien hace GestorDataset(),
        # usamos una bandera para evitar reinicializar los atributos.
        if self._inicializado:
            return
        self._coleccion = ColeccionIris()
        self._ruta_cargada = None
        self._inicializado = True
    
    def cargar(self, ruta_csv):
        """Carga el dataset desde el CSV (solo si no está ya cargado)."""
        if self._ruta_cargada == ruta_csv:
            print(f"[Gestor] El dataset ya está cargado desde '{ruta_csv}'.")
            return
        self._coleccion = ColeccionIris()
        self._coleccion.cargar_desde_csv(ruta_csv)
        self._ruta_cargada = ruta_csv
        print(f"[Gestor] Dataset cargado: {len(self._coleccion)} muestras.")
    
    @property
    def coleccion(self):
        """Acceso a la colección de muestras."""
        return self._coleccion
    
    def info(self):
        return (f"GestorDataset (Singleton) | "
                f"ruta: {self._ruta_cargada} | "
                f"muestras: {len(self._coleccion)}")


print("✅ Clase GestorDataset definida (Singleton).")

In [ ]:
# Probemos que el Singleton funciona

# Crear una primera instancia
g1 = GestorDataset()
g1.cargar(RUTA_CSV)

# Intentar crear otra instancia
g2 = GestorDataset()

# El operador 'is' verifica si dos variables apuntan al MISMO objeto en memoria
print(f"\n¿g1 y g2 son el MISMO objeto? {g1 is g2}")  # True

# Verifiquemos: el dataset ya cargado en g1 también está disponible en g2
print(f"Muestras en g1: {len(g1.coleccion)}")
print(f"Muestras en g2: {len(g2.coleccion)}")

# Si intentamos cargar de nuevo, no recarga
g1.cargar(RUTA_CSV)

**Resultado:** aunque escribimos `GestorDataset()` dos veces, recibimos siempre el mismo objeto. Eso es Singleton.

---
## Paso 7: Demostración final integrada

Pongamos todo a funcionar junto, como en una entrega final. Recorremos cada concepto cubierto.

In [ ]:
def separador(titulo):
    """Imprime un separador visual."""
    print("\n" + "=" * 60)
    print(f"  {titulo}")
    print("=" * 60)


# ========== 1. Singleton ==========
separador("1. PATRÓN SINGLETON")

gestor = GestorDataset()
gestor.cargar(RUTA_CSV)
print(gestor.info())
otro = GestorDataset()
print(f"¿gestor is otro? {gestor is otro}")

coleccion = gestor.coleccion

# ========== 2. Tema 1: muestras y atributos ==========
separador("2. TEMA 1: muestra individual")

primera = coleccion[0]
print("Primera muestra:", primera)
print("Resumen:", primera.resumen())
print(f"Área del sépalo: {primera.area_sepalo_aprox():.2f} cm²")

# ========== 3. Encapsulamiento ==========
separador("3. ENCAPSULAMIENTO")

try:
    primera.sepal_length = -3
except ValueError as e:
    print(f"❌ Medida negativa rechazada: {e}")

try:
    primera.especie = "tulipan"
except ValueError as e:
    print(f"❌ Especie inválida rechazada: {e}")

try:
    primera.id = 999
except AttributeError:
    print("❌ Modificación de Id rechazada (es inmutable).")

# ========== 4. Colección ==========
separador("4. CONSULTAS SOBRE LA COLECCIÓN")

print(f"Total: {len(coleccion)} muestras")
print(f"Setosas:    {len(coleccion.filtrar_por_especie('setosa'))}")
print(f"Versicolor: {len(coleccion.filtrar_por_especie('versicolor'))}")
print(f"Virginica:  {len(coleccion.filtrar_por_especie('virginica'))}")
print(f"Promedio largo de pétalo: {coleccion.promedio_petalo():.2f} cm")

# ========== 5. Herencia y polimorfismo ==========
separador("5. HERENCIA Y POLIMORFISMO")

setosa = coleccion.filtrar_por_especie("setosa")[0]
versicolor = coleccion.filtrar_por_especie("versicolor")[0]
virginica = coleccion.filtrar_por_especie("virginica")[0]

for muestra in [setosa, versicolor, virginica]:
    print(f"  {type(muestra).__name__}: {muestra.clasificar_tamano()}")
    print(f"    {muestra.caracteristicas_especie()}")

# ========== 6. Operadores ==========
separador("6. OPERADORES SOBRECARGADOS")

a = coleccion[0]
b = coleccion[1]
print(f"a = {a}")
print(f"b = {b}")
print(f"a == b: {a == b}")
print(f"a < b:  {a < b}")
print(f"a + b:  {a + b}")

print("\n✅ Demostración completa. Proyecto Unidad 1 finalizado.")

---
## 📋 Resumen de conceptos cubiertos

| Concepto del sílabo | Dónde está | Celda |
|---|---|---|
| Clases y objetos | `MuestraIris`, `ColeccionIris` | Pasos 1, 3 |
| Atributos y métodos | Atributos privados, métodos como `describir`, `area_sepalo_aprox` | Paso 1, 2 |
| Encapsulamiento | Atributos `_` con `@property` y setters | Paso 2 |
| Métodos de acceso | Getters y setters con validación | Paso 2 |
| Herencia | `IrisSetosa`, `IrisVersicolor`, `IrisVirginica` | Paso 4 |
| `super()` | En el `__init__` de cada subclase | Paso 4 |
| Polimorfismo | `clasificar_tamano()` distinto en cada subclase | Paso 4 |
| Métodos mágicos | `__str__`, `__eq__`, `__lt__`, `__len__`, etc. | Paso 5 |
| Sobrecarga de operadores | `+`, `==`, `<`, `in` | Paso 5 |
| Patrón Singleton | `GestorDataset` con `__new__` | Paso 6 |

## 💡 Ideas para extender el proyecto

Si quieres ir más allá:
- Agregar un método `exportar_a_csv()` en `ColeccionIris` para guardar las muestras modificadas.
- Implementar `__sub__` para restar muestras.
- Agregar un método estadístico (mediana, desviación estándar).
- Crear un menú interactivo en consola que use el sistema.